# UK Smart Grid Forecaster - Prophet Training

Use this notebook on Kaggle after uploading `master_training_data.csv` as a Kaggle Dataset. It inspects features, trains Prophet variants, compares validation metrics, and saves the selected model outputs.

In [ ]:
# Run this only if Prophet is not already installed in the Kaggle environment.
try:
    from prophet import Prophet
    print('Prophet already installed')
except ImportError:
    !pip install prophet
    from prophet import Prophet
    print('Prophet installed')

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json

pd.set_option('display.max_columns', 120)

# Change this path after attaching your Kaggle Dataset.
KAGGLE_INPUT_PATH = Path('/kaggle/input/uk-weather/master_training_data.csv')
LOCAL_INPUT_PATH = Path('data/processed/master_training_data.csv')

INPUT_PATH = KAGGLE_INPUT_PATH if KAGGLE_INPUT_PATH.exists() else LOCAL_INPUT_PATH
OUTPUT_FOLDER = Path('/kaggle/working/prophet_outputs') if Path('/kaggle/working').exists() else Path('artifacts/prophet_notebook')
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_FOLDER

In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'demand_mw']).sort_values('timestamp')
df = df.drop_duplicates(subset=['timestamp'], keep='last').reset_index(drop=True)

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('Duplicate timestamps:', df['timestamp'].duplicated().sum())
df.head()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
missing = df[numeric_cols].isna().sum().sort_values(ascending=False)
missing[missing > 0].head(30)

## Candidate Regressors

These are numeric columns that may be useful for Prophet. Text labels and duplicated date columns are excluded.

In [ ]:
candidate_regressors = [
    'temperature_2m',
    'relative_humidity_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'surface_pressure',
    'cloud_cover',
    'wind_speed_10m',
    'shortwave_radiation',
    'weekend',
    'is_holiday',
    'cal_is_bank_holiday_england_wales',
    'cal_is_bank_holiday_scotland',
    'cal_is_event_day',
    'cal_is_non_working_day',
    'cal_is_covid_lockdown',
    'cal_is_general_election',
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',
    'econ_economic_data_complete',
]

candidate_regressors = [c for c in candidate_regressors if c in df.columns]
for col in candidate_regressors:
    df[col] = pd.to_numeric(df[col], errors='coerce').ffill().bfill()

candidate_regressors

In [ ]:
validation_days = 30
validation_start = df['timestamp'].max() - pd.Timedelta(days=validation_days) + pd.Timedelta(hours=1)
train_df = df[df['timestamp'] < validation_start].copy()
valid_df = df[df['timestamp'] >= validation_start].copy()

print('Train:', len(train_df), train_df['timestamp'].min(), 'to', train_df['timestamp'].max())
print('Valid:', len(valid_df), valid_df['timestamp'].min(), 'to', valid_df['timestamp'].max())

In [ ]:
feature_report = []
for col in candidate_regressors:
    if train_df[col].nunique(dropna=True) <= 1:
        continue
    feature_report.append({
        'feature': col,
        'unique_values': train_df[col].nunique(dropna=True),
        'train_corr': train_df[[col, 'demand_mw']].corr().iloc[0, 1],
        'valid_corr': valid_df[[col, 'demand_mw']].corr().iloc[0, 1],
        'missing_train': train_df[col].isna().sum(),
        'missing_valid': valid_df[col].isna().sum(),
    })

feature_report = pd.DataFrame(feature_report)
feature_report['abs_valid_corr'] = feature_report['valid_corr'].abs()
feature_report = feature_report.sort_values('abs_valid_corr')
feature_report.to_csv(OUTPUT_FOLDER / 'feature_strength.csv', index=False)
feature_report

## Prophet Helpers

In [ ]:
def make_prophet_frame(source_df, regressors):
    out = source_df[['timestamp', 'demand_mw', *regressors]].copy()
    out = out.rename(columns={'timestamp': 'ds', 'demand_mw': 'y'})
    return out


def calculate_metrics(actual, predicted):
    actual = pd.Series(actual).reset_index(drop=True)
    predicted = pd.Series(predicted).reset_index(drop=True)
    error = actual - predicted
    ss_res = float((error ** 2).sum())
    ss_tot = float(((actual - actual.mean()) ** 2).sum())
    return {
        'mae': round(float(error.abs().mean()), 4),
        'rmse': round(float((error ** 2).mean() ** 0.5), 4),
        'mape': round(float((error.abs() / actual.abs().clip(lower=1)).mean() * 100), 4),
        'r2': round(float(1 - ss_res / ss_tot), 4) if ss_tot else 0.0,
    }


def train_prophet_variant(name, params, regressors):
    train = make_prophet_frame(train_df, regressors)
    valid = make_prophet_frame(valid_df, regressors)
    regressors = [c for c in regressors if train[c].nunique(dropna=True) > 1]

    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode=params.get('seasonality_mode', 'additive'),
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.05),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
    )
    model.add_seasonality('daily', period=1, fourier_order=params.get('daily_fourier_order', 16))
    model.add_seasonality('weekly', period=7, fourier_order=params.get('weekly_fourier_order', 10))
    model.add_seasonality('yearly', period=365.25, fourier_order=params.get('yearly_fourier_order', 12))
    model.add_country_holidays(country_name='UK')

    for regressor in regressors:
        model.add_regressor(regressor)

    print('Training', name, 'with', len(regressors), 'regressors')
    model.fit(train[['ds', 'y', *regressors]])
    forecast = model.predict(valid[['ds', *regressors]])
    predictions = valid[['ds', 'y']].merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='left')
    metrics = calculate_metrics(predictions['y'], predictions['yhat'])
    return {'name': name, 'model': model, 'predictions': predictions, 'metrics': metrics, 'regressors': regressors, 'params': params}


## Train Variants

Start with the current best structure, then test reduced-feature variants.

In [ ]:
full_regressors = candidate_regressors

reduced_regressors = [
    'temperature_2m',
    'apparent_temperature',
    'precipitation',
    'cloud_cover',
    'shortwave_radiation',
    'weekend',
    'is_holiday',
    'cal_is_non_working_day',
]
reduced_regressors = [c for c in reduced_regressors if c in df.columns]

weather_only_regressors = [
    'temperature_2m',
    'apparent_temperature',
    'precipitation',
    'cloud_cover',
    'shortwave_radiation',
    'weekend',
]
weather_only_regressors = [c for c in weather_only_regressors if c in df.columns]

variants = [
    ('full_v1_like', {
        'daily_fourier_order': 16,
        'weekly_fourier_order': 10,
        'yearly_fourier_order': 12,
        'changepoint_prior_scale': 0.10,
        'seasonality_mode': 'additive',
    }, full_regressors),
    ('reduced_features', {
        'daily_fourier_order': 16,
        'weekly_fourier_order': 10,
        'yearly_fourier_order': 12,
        'changepoint_prior_scale': 0.10,
        'seasonality_mode': 'additive',
    }, reduced_regressors),
    ('weather_only', {
        'daily_fourier_order': 16,
        'weekly_fourier_order': 10,
        'yearly_fourier_order': 12,
        'changepoint_prior_scale': 0.10,
        'seasonality_mode': 'additive',
    }, weather_only_regressors),
]

results = []
for name, params, regressors in variants:
    result = train_prophet_variant(name, params, regressors)
    print(name, result['metrics'])
    results.append(result)


In [ ]:
comparison = pd.DataFrame([
    {'model': r['name'], **r['metrics'], 'regressor_count': len(r['regressors'])}
    for r in results
]).sort_values('mape')

comparison.to_csv(OUTPUT_FOLDER / 'model_comparison.csv', index=False)
comparison

In [ ]:
best = min(results, key=lambda r: r['metrics']['mape'])
best['predictions'].to_csv(OUTPUT_FOLDER / 'validation_predictions.csv', index=False)

with open(OUTPUT_FOLDER / 'prophet_model.json', 'w', encoding='utf-8') as f:
    f.write(model_to_json(best['model']))

summary = {
    'selected_model': best['name'],
    'metrics': best['metrics'],
    'regressors': best['regressors'],
    'params': best['params'],
    'train_rows': len(train_df),
    'validation_rows': len(valid_df),
}
with open(OUTPUT_FOLDER / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

summary